# Notebook 1: FRIP Signal Generation (GEE)

This notebook computes the raw Flooding Role in Productivity (FRIP) signal using Google Earth Engine.
It follows a strict modular structure:
1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Visual Integration Test
5. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee
import geemap

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")

# Output destination
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
STUDY_REGION = ee.FeatureCollection([
    ee.Feature(CONGO_BBOX, {'basin': 'Congo'}),
    ee.Feature(AMAZON_BBOX, {'basin': 'Amazon'})
])

# Scales (meters)
SCALES = list(range(5000, 105000, 5000))

# Datasets
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
MERIT_HYDRO = 'MERIT/Hydro/v1_0_1'
GLOFAS = 'JRC/CEMS_GLOFAS/FloodHazard/v2_1'
MODIS_NPP = 'MODIS/061/MOD17A3HGF'
YEARS = list(range(2001, 2024))

print("✓ Configuration loaded.")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

def load_and_mask_base_layers():
    """Loads and masks the JRC TMF and MERIT Hydro connectivity layers."""
    tmf = ee.ImageCollection(FOREST_MASK).mosaic()
    forest_mask = tmf.eq(FOREST_CLASS)
    
    merit = ee.Image(MERIT_HYDRO)
    hnd_mask = merit.select('hnd').gt(0)
    
    combined_mask = forest_mask.updateMask(hnd_mask)
    return combined_mask

def load_frip_inputs(combined_mask):
    """Loads MODIS NPP and GLOFAS Flood Depth, applying the combined mask."""
    glofas = ee.ImageCollection(GLOFAS)
    depth_bands = ['RP10_depth', 'RP20_depth', 'RP50_depth', 'RP75_depth', 'RP100_depth', 'RP200_depth', 'RP500_depth']
    flood_depth = glofas.mosaic().select(depth_bands).reduce(ee.Reducer.sum()).rename('depth').updateMask(combined_mask)
    
    modis = ee.ImageCollection(MODIS_NPP).select('Npp')
    def get_annual(year):
        img = modis.filter(ee.Filter.calendarRange(year, year, 'year')).first()
        return img.updateMask(combined_mask).set('year', year)
    
    annual_npp = ee.ImageCollection.fromImages(ee.List(YEARS).map(get_annual))
    mean_npp = annual_npp.mean()
    npp_proj = annual_npp.first().projection()
    
    return flood_depth, annual_npp, mean_npp, npp_proj

def compute_spatial_correlation(npp_img, flood_depth, scale, npp_proj):
    """Computes spatial Spearman correlation between NPP and Flood Depth within a grid cell."""
    stack = ee.Image.cat([npp_img.rename('npp'), flood_depth.rename('depth')]).setDefaultProjection(npp_proj)
    
    # reduceResolution computes the correlation of native pixels within the new coarse pixel
    spearman = stack.reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=65535
    ).reproject(
        crs=npp_proj,
        scale=scale
    )
    
    return spearman.select('correlation')

def build_frip_assets(scale):
    """Orchestrates the computation for a given scale, returning cross-sectional and annual FRIP images."""
    combined_mask = load_and_mask_base_layers()
    flood_depth, annual_npp, mean_npp, npp_proj = load_frip_inputs(combined_mask)
    
    # Cross-sectional FRIP
    frip_cross = compute_spatial_correlation(mean_npp, flood_depth, scale, npp_proj).rename(f'FRIP_{scale}')
    
    # Annual FRIP
    def compute_annual(img):
        year = ee.Number(img.get('year')).format('%04d')
        corr = compute_spatial_correlation(img, flood_depth, scale, npp_proj)
        return corr.rename(ee.String('FRIP_').cat(year))
    
    frip_annual_col = annual_npp.map(compute_annual)
    frip_annual_img = frip_annual_col.toBands()
    
    # Clean up band names for annual image (remove the list index prefix)
    band_names = [f'FRIP_{y}' for y in YEARS]
    frip_annual_img = frip_annual_img.rename(band_names)
    
    return frip_cross, frip_annual_img, npp_proj

print("✓ Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running Unit Tests for FRIP generation...")
    test_scale = 50000
    try:
        frip_cross, frip_annual, proj = build_frip_assets(test_scale)
        
        # Test 1: Image types
        assert isinstance(frip_cross, ee.Image), "Cross-sectional output is not an ee.Image"
        assert isinstance(frip_annual, ee.Image), "Annual output is not an ee.Image"
        
        # Test 2: Band names
        cross_bands = frip_cross.bandNames().getInfo()
        assert len(cross_bands) == 1, "Cross-sectional image should have exactly 1 band"
        assert cross_bands[0] == f'FRIP_{test_scale}', f"Expected band name FRIP_{test_scale}"
        
        annual_bands = frip_annual.bandNames().getInfo()
        assert len(annual_bands) == len(YEARS), f"Expected {len(YEARS)} annual bands, got {len(annual_bands)}"
        assert annual_bands[0] == 'FRIP_2001', "First annual band should be FRIP_2001"
        assert annual_bands[-1] == 'FRIP_2023', "Last annual band should be FRIP_2023"
        
        print("✓ Basic structure tests passed. Running deep execution test (this takes ~10 seconds)...")
        
        # Test 3: Deep execution (ensure graph evaluates without GEE errors)
        # We sample a tiny 50x50km bounding box in the Congo to test computation
        tiny_test_region = ee.Geometry.Rectangle([15.0, 0.0, 15.5, 0.5])
        test_val = frip_cross.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=tiny_test_region,
            scale=test_scale,
            maxPixels=1e9
        ).getInfo()
        
        assert isinstance(test_val, dict), "Reduction did not return a dictionary"
        assert f'FRIP_{test_scale}' in test_val, "Value dictionary missing target band"
        
        print(f"✓ Deep execution passed! Sample FRIP mean over test region: {test_val[f'FRIP_{test_scale}']}")
        print("✓ All unit tests passed successfully!")
        
    except AssertionError as e:
        print(f"✗ Unit Test Failed: {e}")
    except Exception as e:
        print(f"✗ Unexpected Error during tests: {e}")

# Execute tests
run_unit_tests()


In [ ]:
# =============================================================================
# BLOCK 4: VISUAL INTEGRATION TEST
# =============================================================================

def visualize_demo():
    print("Generating interactive map for visual inspection (50km scale)...")
    test_scale = 50000
    frip_cross, frip_annual, proj = build_frip_assets(test_scale)
    
    Map = geemap.Map(center=[0, 20], zoom=3)
    
    # Visualization parameters for Spearman correlation [-1 to 1]
    vis_params = {
        'min': -1,
        'max': 1,
        'palette': ['red', 'white', 'green']
    }
    
    Map.addLayer(frip_cross, vis_params, 'FRIP Cross-sectional (50km)')
    Map.addLayer(STUDY_REGION, {'color': 'blue'}, 'Study Regions', False)
    
    return Map

# Display the map
visualize_demo()

In [ ]:
# =============================================================================
# BLOCK 5: EXECUTION (ASSET EXPORT)
# =============================================================================

def export_all_scales(dry_run=True):
    tasks = []
    print(f"Configuring export tasks for {len(SCALES)} scales...")
    
    for scale in SCALES:
        frip_cross, frip_annual, proj = build_frip_assets(scale)
        
        # Cross-sectional export
        task_cross = ee.batch.Export.image.toAsset(
            image=frip_cross,
            description=f'FRIP_{scale}_Export',
            assetId=f'{ASSET_ROOT}/FRIP_{scale}',
            region=STUDY_REGION.geometry(),
            scale=scale,
            crs=proj,
            maxPixels=1e13
        )
        tasks.append(task_cross)
        
        # Annual export
        task_ann = ee.batch.Export.image.toAsset(
            image=frip_annual,
            description=f'FRIP_Annual_{scale}_Export',
            assetId=f'{ASSET_ROOT}/FRIP_Annual_{scale}',
            region=STUDY_REGION.geometry(),
            scale=scale,
            crs=proj,
            maxPixels=1e13
        )
        tasks.append(task_ann)
        
        if not dry_run:
            task_cross.start()
            task_ann.start()
            
    print(f"✓ Configured {len(tasks)} export tasks.")
    if dry_run:
        print("DRY RUN: Tasks created but not started. Call export_all_scales(dry_run=False) to begin processing on GEE servers.")
    else:
        print("Tasks started! Monitor progress in the GEE Code Editor or via ee.batch.Task.list()")

# To execute the exports, set dry_run=False
export_all_scales(dry_run=True)